# 02 - Multi-Table Relational EDA & Aggregations

**Track 2 Lead**: Open / In Progress  
**Course**: UC Berkeley DATASCI 207 (Machine Learning at Scale)  
**Target Deliverable**: Project Milestone & Final Submission  
**Scope**: Multi-table relational joins across `bureau.csv`, `previous_application.csv`, `POS_CASH_balance.csv`, `installments_payments.csv`, and `credit_card_balance.csv`.

---

### Environment & Data Ingestion (Google Colab)
Run the cell below if executing in Google Colab to mount Google Drive and pull dataset files to `/content/local_data`.


In [1]:
# NOTE FOR TEAMMATES:
# If you get 'Directory not found', add the shared folder to your My Drive:
# 1. Go to Google Drive (https://drive.google.com).
# 2. Click 'Shared with me' on the left menu.
# 3. Right-click the 'DATASCI207_finalproject' folder.
# 4. Select Organize -> Add shortcut (or press Shift + Z).
# 5. Choose 'My Drive' and click Add.

try:
    from google.colab import drive
    import shutil, os

    # Mount Google Drive
    drive.mount('/content/drive')

    # Copy shared project folder to local Colab storage
    src = "/content/drive/MyDrive/DATASCI207_finalproject"
    dst = "/content/DATASCI207_finalproject"

    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.gdoc', '*.gsheet', '*.gslides'))
    print("Downloaded files:", os.listdir(dst))
except ImportError:
    print("Running in local environment; skipping Google Drive mount.")


Running in local environment; skipping Google Drive mount.


### Cleaned Static Baseline Ingestion (Task 1)

Loads the cleaned and engineered static applicant dataset (`application_train_cleaned.parquet`) produced by Task 1.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve repo root regardless of where the kernel's cwd ends up
# (Jupyter often sets cwd to the notebook's own folder, i.e. notebooks/)
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

# 1. Load Task 1 cleaned application matrix
data_path = Path('/content/DATASCI207_finalproject/data/processed/application_train_cleaned.parquet')
if not data_path.exists():
    data_path = PROJECT_ROOT / 'data' / 'processed' / 'application_train_cleaned.parquet'

df_cleaned = pd.read_parquet(data_path)
print(f'Loaded cleaned data: {df_cleaned.shape[0]:,} rows x {df_cleaned.shape[1]:,} columns')

# 2. Raw relational tables directory (bureau, previous_application, etc.)
raw_dir = Path('/content/DATASCI207_finalproject/data/raw/home_credit_default_risk')
if not raw_dir.exists():
    raw_dir = PROJECT_ROOT / 'data' / 'raw' / 'home_credit_default_risk'
RAW_DIR = raw_dir
if RAW_DIR.exists():
    print('Available raw tables:', [f.name for f in RAW_DIR.glob('*.csv')])

Loaded cleaned data: 307,511 rows x 131 columns
Available raw tables: ['application_test.csv', 'HomeCredit_columns_description.csv', 'POS_CASH_balance.csv', 'credit_card_balance.csv', 'installments_payments.csv', 'application_train.csv', 'bureau.csv', 'previous_application.csv', 'bureau_balance.csv', 'sample_submission.csv']


In [3]:
import gc

def downcast(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')
    return df

def mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1e6



In [4]:
## aggregation 1 - bureau.csv - external credit bureau lines
bureau = pd.read_csv(
    RAW_DIR / 'bureau.csv',
    usecols=['SK_ID_CURR', 'CREDIT_ACTIVE', 'AMT_CREDIT_SUM_OVERDUE',
             'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM_DEBT', 'DAYS_CREDIT']
)
print(f"bereau.csv loaded: {bureau.shape[0]:,} rows, {mb(bureau):.1f} MB")
bureau = downcast(bureau)
print(f"after downcast: {mb(bureau):.1f} MB")

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_ACTIVE_LINES=('CREDIT_ACTIVE', lambda s: (s == 'Active').sum()),
    BUREAU_TOTAL_LINES=('CREDIT_ACTIVE', 'count'),
    BUREAU_TOTAL_OVERDUE_DEBT=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    BUREAU_MAX_DAYS_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_OLDEST_CREDIT_DAYS=('DAYS_CREDIT', 'min'),
).reset_index()

del bureau
gc.collect()
bureau_agg.head()

bereau.csv loaded: 1,716,428 rows, 92.7 MB
after downcast: 42.9 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_24869/1900776898.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,BUREAU_ACTIVE_LINES,BUREAU_TOTAL_LINES,BUREAU_TOTAL_OVERDUE_DEBT,BUREAU_MAX_DAYS_OVERDUE,BUREAU_TOTAL_DEBT,BUREAU_OLDEST_CREDIT_DAYS
0,100001,3,7,0.0,0,596686.5,-1572
1,100002,2,8,0.0,0,245781.0,-1437
2,100003,1,4,0.0,0,0.0,-2586
3,100004,0,2,0.0,0,0.0,-1326
4,100005,2,3,0.0,0,568408.5,-373


In [5]:
## aggregation 2 - previous_application.csv - prior home credit applications
prev = pd.read_csv(
    RAW_DIR / 'previous_application.csv',
    usecols=['SK_ID_CURR', 'NAME_CONTRACT_STATUS', 'AMT_DOWN_PAYMENT', 'AMT_CREDIT']
)
print(f"previous_application.csv loaded: {prev.shape[0]:,} rows, {mb(prev):.1f} MB")
prev = downcast(prev)

prev['DOWN_PAYMENT_PCT'] = prev['AMT_DOWN_PAYMENT'] / prev['AMT_CREDIT']
prev['DOWN_PAYMENT_PCT'] = prev['DOWN_PAYMENT_PCT'].replace([np.inf, -np.inf], np.nan)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_REFUSAL_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Refused').sum()),
    PREV_APPLICATION_COUNT=('NAME_CONTRACT_STATUS', 'count'),
    PREV_APPROVAL_RATE=('NAME_CONTRACT_STATUS', lambda s: (s == 'Approved').mean()),
    PREV_MEAN_DOWN_PAYMENT_PCT=('DOWN_PAYMENT_PCT', 'mean'),
).reset_index()

del prev
gc.collect()
prev_agg.head()

previous_application.csv loaded: 1,670,214 rows, 66.6 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_24869/1900776898.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,PREV_REFUSAL_COUNT,PREV_APPLICATION_COUNT,PREV_APPROVAL_RATE,PREV_MEAN_DOWN_PAYMENT_PCT
0,100001,0,1,1.0,0.105940
1,100002,0,1,1.0,0.000000
2,100003,0,3,1.0,0.050585
3,100004,0,1,1.0,0.241719
4,100005,0,2,0.5,0.111173


In [6]:
## aggregation 3 - POS_CASH.csv - monthly POS_CASH balances
pos = pd.read_csv(
    RAW_DIR / 'POS_CASH_balance.csv',
    usecols=['SK_ID_CURR', 'MONTHS_BALANCE', 'SK_DPD', 'SK_DPD_DEF', 'NAME_CONTRACT_STATUS',]
)
print(f"POS_CASH_balance.csv loaded: {pos.shape[0]:,} rows, {mb(pos):.1f} MB")
pos = downcast(pos)

pos_agg = pos.groupby('SK_ID_CURR').agg(
    POS_MONTH_COUNT=('MONTHS_BALANCE', 'count'), # trajectory length
    POS_MOST_RECENT_MONTH=('MONTHS_BALANCE', 'count'), # closest to 0 = most recent
    POS_MEAN_DPD=('SK_DPD', 'mean'),
    POS_MAX_DPD=('SK_DPD', 'max'),
    POS_MEAN_DPD_DEF=('SK_DPD_DEF', 'mean'),
    POS_COMPLETED_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Completed').sum()),
    POS_ACTIVE_COUNT=('NAME_CONTRACT_STATUS', lambda s: (s == 'Active').sum()),
).reset_index()

del pos
gc.collect()
pos_agg.head()

POS_CASH_balance.csv loaded: 10,001,358 rows, 462.4 MB


/var/folders/y4/6v54dls90g9_fpz5vx1pzmnh0000gn/T/ipykernel_24869/1900776898.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


,SK_ID_CURR,POS_MONTH_COUNT,POS_MOST_RECENT_MONTH,POS_MEAN_DPD,POS_MAX_DPD,POS_MEAN_DPD_DEF,POS_COMPLETED_COUNT,POS_ACTIVE_COUNT
0,100001,9,9,0.777778,7,0.777778,2,7
1,100002,19,19,0.000000,0,0.000000,0,19
2,100003,28,28,0.000000,0,0.000000,2,26
3,100004,4,4,0.000000,0,0.000000,1,3
4,100005,11,11,0.000000,0,0.000000,1,9


In [7]:
## aggregation 4 - installments_payments.csv - big data set
inst = pd.read_csv(
    RAW_DIR / 'installments_payments.csv',
    usecols=['SK_ID_CURR', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT', 'AMT_INSTALMENT', 'AMT_PAYMENT']
)
print(f"installments_payments.csv loaded: {inst.shape[0]:,} rows, {mb(inst):.1f} MB")
inst = downcast(inst)

# positive DELAY_DAYS = paid late - negative/zero = on time or early
inst['DELAY_DAYS'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['UNDERPAID'] = inst['AMT_PAYMENT'] < inst['AMT_INSTALMENT']

inst_agg = inst.groupby('SK_ID_CURR').agg(
    INST_DELAY_MEAN=('DELAY_DAYS', 'mean'),
    INST_DELAY_MAX=('DELAY_DAYS', 'max'),
    INST_UNDERPAYMENT_FREQ=('UNDERPAID', 'mean'), # fraction of installments underpaid
    INST_TOTAL_PAYMENTS=('AMT_PAYMENT', 'count'),
).reset_index()

del inst
gc.collect()
inst_agg.head()

installments_payments.csv loaded: 13,605,401 rows, 544.2 MB


,SK_ID_CURR,INST_DELAY_MEAN,INST_DELAY_MAX,INST_UNDERPAYMENT_FREQ,INST_TOTAL_PAYMENTS
0,100001,-7.285714,11.0,0.0,7
1,100002,-20.421053,-12.0,0.0,19
2,100003,-7.160000,-1.0,0.0,25
3,100004,-7.666667,-3.0,0.0,3
4,100005,-23.555555,1.0,0.0,9


In [8]:
## join with SK_ID_CURR + merge onto the clean table
print(f"Base (Task 1) shape: {df_cleaned.shape}, {mb(df_cleaned):.1f} MB")

processed_features = (
    df_cleaned.merge(bureau_agg, on='SK_ID_CURR', how='left')
                .merge(prev_agg, on='SK_ID_CURR', how='left')
                .merge(pos_agg, on='SK_ID_CURR', how='left')
                .merge(inst_agg, on='SK_ID_CURR', how='left')
)

## rows with no match in behavioral table are NaN after the left join
## for *counts*, NaN means "zero history" -> fill with 0
count_cols = [c for c in processed_features.columns if c.endswith('_COUNT') or c.endswith('LINES')]
processed_features[count_cols] = processed_features[count_cols].fillna(0)

processed_features['HAS_BUREAU_HISTORY'] = (processed_features['BUREAU_TOTAL_LINES'] > 0).astype('int8')

print(f"Final merged shape: {processed_features.shape}, {mb(processed_features):.1f} MB")

Base (Task 1) shape: (307511, 131), 361.0 MB
Final merged shape: (307511, 153), 410.5 MB


In [9]:
## export to compressed parquet + document size reduction
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'processed_features.parquet'
processed_features.to_parquet(OUTPUT_PATH, engine='pyarrow', compression='snappy', index=False)

output_size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f"Exported {processed_features.shape[0]:,} rows x {processed_features.shape[1]} cols "
      f"-> {OUTPUT_PATH} ({output_size_mb:.1f} MB on disk)")

Exported 307,511 rows x 153 cols -> /Users/tungla/Desktop/UC Berkeley/DATASCI207_AppliedMachineLearning/DATASCI-207-FinalProject-Team1/DATASCI-207-FinalProject-Team1/data/processed/processed_features.parquet (37.3 MB on disk)
